In [1]:
import pandas as pd
import re
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

# Load datasets
macro_dataset_path = 'E:\\Economic_Data\\Input data\\Splitting Economic data\\Macro Dataset 1.csv'
merged_dataset_path = 'E:\\Economic_Data\\Input data\\merged-dataset\\merged_datasets.csv'
price_dataset_path = 'E:\\Economic_Data\\Input data\\price.csv'

macro_df = pd.read_csv(macro_dataset_path)
merged_df = pd.read_csv(merged_dataset_path)
price_df = pd.read_csv(price_dataset_path)

# Ensure datetime columns are in datetime format
merged_df['datetime'] = pd.to_datetime(merged_df['datetime'], errors='coerce')
merged_df.dropna(subset=['datetime'], inplace=True)

# Clean event names
def clean_event_name(event_name):
    return re.sub(r'\((?!WoW|YoY|MoM).*?\)', '', event_name).strip()

merged_df['Cleaned_Event'] = merged_df['Event'].apply(clean_event_name)
filtered_merged_df = merged_df[merged_df['Cleaned_Event'].isin(macro_df['Event'])]

# Convert columns to string type before cleaning
filtered_merged_df['Actual'] = filtered_merged_df['Actual'].astype(str)
filtered_merged_df['Forecast'] = filtered_merged_df['Forecast'].astype(str)
filtered_merged_df['Previous'] = filtered_merged_df['Previous'].astype(str)

# Function to clean numeric columns
def clean_numeric_column(column):
    return pd.to_numeric(
        column.str.replace('%', '')
               .str.replace('M', 'e6')
               .str.replace('B', 'e9')
               .str.replace('K', 'e3')
               .str.replace('[^\d.e]', '', regex=True),
        errors='coerce'
    ).fillna(0)

filtered_merged_df['Actual'] = clean_numeric_column(filtered_merged_df['Actual'])
filtered_merged_df['Forecast'] = clean_numeric_column(filtered_merged_df['Forecast'])
filtered_merged_df['Previous'] = clean_numeric_column(filtered_merged_df['Previous'])

# Calculate differences
filtered_merged_df['Actual_Previous_Diff'] = filtered_merged_df['Actual'] - filtered_merged_df['Previous']
filtered_merged_df['Actual_Forecast_Diff'] = filtered_merged_df['Actual'] - filtered_merged_df['Forecast']

# Ensure differences are numeric
filtered_merged_df['Actual_Previous_Diff'] = pd.to_numeric(filtered_merged_df['Actual_Previous_Diff'], errors='coerce')
filtered_merged_df['Actual_Forecast_Diff'] = pd.to_numeric(filtered_merged_df['Actual_Forecast_Diff'], errors='coerce')

price_df['datetime'] = pd.to_datetime(price_df['datetime'])

# Merge datasets on datetime
merged_data = pd.merge_asof(filtered_merged_df.sort_values('datetime'), 
                            price_df[['datetime', 'close']].sort_values('datetime'), 
                            on='datetime', 
                            direction='backward')

# Function to calculate percentage change
def calculate_percentage_change(current_price, future_price):
    return ((future_price - current_price) / current_price) * 100

time_deltas = [5, 15, 30, 60]
for delta in time_deltas:
    future_price_df = price_df[['datetime', 'close']].copy()
    future_price_df['datetime'] = future_price_df['datetime'] - pd.Timedelta(minutes=delta)
    future_price_df.rename(columns={'close': f'close_price_{delta}m'}, inplace=True)
    merged_data = pd.merge_asof(merged_data.sort_values('datetime'), 
                                future_price_df.sort_values('datetime'), 
                                on='datetime', 
                                direction='forward')

for delta in time_deltas:
    merged_data[f'pct_change_{delta}m'] = calculate_percentage_change(merged_data['close'], merged_data[f'close_price_{delta}m'])

# Function to perform PCA Ridge Regression analysis
def pca_ridge_regression_analysis(event_data, y_column):
    X = event_data[['Actual_Previous_Diff', 'Actual_Forecast_Diff']]
    y = event_data[y_column]

    # Create a pipeline with standard scaler, PCA, and ridge regression
    pca_ridge_model = make_pipeline(StandardScaler(), PCA(n_components=2), Ridge(alpha=1.0))

    # Fit the model
    pca_ridge_model.fit(X, y)

    # Get the coefficients
    ridge = pca_ridge_model.named_steps['ridge']
    coefficients = ridge.coef_
    intercept = ridge.intercept_
    r_squared = pca_ridge_model.score(X, y)

    return coefficients, intercept, r_squared

# Loop through each group and perform PCA Ridge Regression analysis for each time delta
results = []

for event, event_data in merged_data.groupby('Cleaned_Event'):
    print(f"Processing event: {event}")  # Debug print

    for delta in time_deltas:
        y_column = f'pct_change_{delta}m'
        
        valid_data = event_data.dropna(subset=['Actual_Previous_Diff', 'Actual_Forecast_Diff', y_column])

        if not valid_data.empty:
            coefficients, intercept, r_squared = pca_ridge_regression_analysis(valid_data, y_column)
            result = {
                'Event': event,
                'Time_Delta': delta,
                'Coefficients': coefficients,
                'Intercept': intercept,
                'R_squared': r_squared
            }
            results.append(result)

# Convert results to DataFrame for easier viewing
results_df = pd.DataFrame(results)

# Save results to a CSV file
results_file_path = 'E:\\Economic_Data\\Output data\\modelling output\\pca_ridge_regression_results2.csv'
results_df.to_csv(results_file_path, index=False)

print(f"Results saved to {results_file_path}")


C:\Users\Zeinab\AppData\Local\Temp\ipykernel_21116\1724856981.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_merged_df['Actual'] = filtered_merged_df['Actual'].astype(str)
C:\Users\Zeinab\AppData\Local\Temp\ipykernel_21116\1724856981.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_merged_df['Forecast'] = filtered_merged_df['Forecast'].astype(str)
C:\Users\Zeinab\AppData\Local\Temp\ipykernel_21116\1724856981.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of

Processing event: Atlanta Fed GDPNow
Processing event: Average Hourly Earnings (MoM)
Processing event: Average Hourly Earnings (YoY) (YoY)
Processing event: Average Weekly Hours
Processing event: Building Permits
Processing event: Building Permits (MoM)
Processing event: Business Inventories (MoM)
Processing event: CB Consumer Confidence
Processing event: CFTC Aluminium speculative net positions
Processing event: CFTC Copper speculative net positions
Processing event: CFTC Corn speculative net positions
Processing event: CFTC Crude Oil speculative net positions
Processing event: CFTC Gold speculative net positions
Processing event: CFTC Nasdaq 100 speculative net positions
Processing event: CFTC Natural Gas speculative net positions
Processing event: CFTC S&P 500 speculative net positions
Processing event: CFTC Silver speculative net positions
Processing event: CFTC Soybeans speculative net positions
Processing event: CFTC Wheat speculative net positions
Processing event: CPI (MoM)
Pro